In [1]:
import cv2
import json
import time
import torch
import random
import numpy as np
import matplotlib.pyplot as plt

from tqdm import tqdm
from PIL import Image
from transformers import Qwen2TokenizerFast, Qwen3ForCausalLM, AutoConfig

from diffusers.utils import load_image
from diffusers.schedulers import FlowMatchEulerDiscreteScheduler
from diffusers.models import AutoencoderKLFlux2, Flux2Transformer2DModel

from ctgmworkshop.flux_tools import *
from ctgmworkshop.image_tools import *
from ctgmworkshop.optical_flow_tools import *
from ctgmworkshop.architectures.flow_refractor_unet_2 import UNet
from ctgmworkshop.tiny_vae import DiffusersTAEF2Wrapper

device = "cuda"
dtype = torch.bfloat16
plt.style.use("dark_background")

In [2]:
text_encoder = Qwen3ForCausalLM.from_pretrained(
    "../../FLUX.2-klein-4B/text_encoder", local_files_only=True
).to(device, dtype)
tokenizer = Qwen2TokenizerFast.from_pretrained(
    "../../FLUX.2-klein-4B/tokenizer", local_files_only=True, device=device
)

scheduler = FlowMatchEulerDiscreteScheduler.from_pretrained(
    "../../FLUX.2-klein-4B/scheduler", device=device
)
vae = AutoencoderKLFlux2.from_pretrained("../../FLUX.2-klein-4B/vae", device=device).to(
    device, dtype
)
transformer = Flux2Transformer2DModel.from_pretrained(
    "../../FLUX.2-klein-4B/transformer", device=device
).to(device, dtype)

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

In [3]:
height = 512
width = 960

In [4]:
@torch.no_grad()
def process_frame(frame, seed, prompt):
    num_inference_steps = 2

    frame = norm_image(resize_torch(cv2_to_torch(frame), height=height, width=width))

    image_latents, image_latent_ids = prepare_image_latents(
        images=[frame], vae=vae, batch_size=1, device=device, dtype=dtype
    )

    cond_latents = unpack_latents_with_ids(image_latents, image_latent_ids)
    cond_latents = unpatchify_latents(cond_latents)

    # cond_latents -> return

    prompt_embeds, text_ids = encode_prompt(
        prompt=prompt,
        text_encoder=text_encoder,
        tokenizer=tokenizer,
        device=device,
        num_images_per_prompt=1,
        max_sequence_length=512,
        text_encoder_out_layers=(9, 18, 27),
    )

    generator = torch.Generator(device="cuda").manual_seed(seed)

    latents, latent_ids = prepare_latents(
        batch_size=1,
        num_latents_channels=32,
        height=height,
        width=width,
        dtype=dtype,
        device=device,
        generator=generator,
    )

    sigmas = np.linspace(1.0, 1 / num_inference_steps, num_inference_steps)

    image_seq_len = latents.shape[1]

    mu = compute_empirical_mu(
        image_seq_len=image_seq_len, num_steps=num_inference_steps
    )

    timesteps, num_inference_steps = retrieve_timesteps(
        scheduler,
        num_inference_steps,
        device,
        sigmas=sigmas,
        mu=mu,
    )

    scheduler.set_begin_index(0)

    init_latents = latents.clone()
    init_latents = unpack_latents_with_ids(init_latents, latent_ids)
    init_latents = unpatchify_latents(init_latents)

    # init_latents -> return

    for i, t in enumerate(timesteps):
        timestep = t.expand(latents.shape[0]).to(latents.dtype)

        latent_model_input = latents.to(transformer.dtype)
        latent_image_ids = latent_ids

        noise_pred = transformer(
            hidden_states=torch.cat([latent_model_input, image_latents], dim=1),
            timestep=timestep / 1000,
            guidance=None,
            encoder_hidden_states=prompt_embeds,
            txt_ids=text_ids,
            img_ids=torch.cat([latent_image_ids, image_latent_ids], dim=1),
            return_dict=False,
        )[0]

        noise_pred = noise_pred[:, : latents.size(1), :]

        latents = scheduler.step(noise_pred, t, latents, return_dict=False)[0]

    vae_scale_factor = 8

    latent_height = 2 * (int(height) // (vae_scale_factor * 2))
    latent_width = 2 * (int(width) // (vae_scale_factor * 2))

    latents = unpack_latents_with_ids(
        latents, latent_ids, latent_height // 2, latent_width // 2
    )

    final_latents = unpatchify_latents(latents)

    # final_latents -> return

    return cond_latents, init_latents, final_latents


In [32]:
unet = UNet()
unet = unet.to(device)

optimizer = torch.optim.AdamW(unet.parameters(), lr=1e-4)
num_train_timesteps = 1000

total_params = sum(p.numel() for p in unet.parameters())
print("Total params:", total_params / 1000000, "M")

state_dict = torch.load(
    "../../trained_models/flow_refractor/test-6-rec.pth", weights_only=True
)
unet.load_state_dict(state_dict)

Total params: 93.598496 M


<All keys matched successfully>

In [45]:
@torch.no_grad()
def refract(init, cond_A, cond_B, final_A):
    conditioning = torch.cat([cond_A, cond_B, final_A], dim=1)
    scheduler.set_timesteps(4, mu=1.0)
    latents = torch.normal(0, 1, init.shape).to(init.device, init.dtype)
    unet.eval()

    for t in scheduler.timesteps:
        latent_model_input = latents
        t = t.to(device).view(1)
        predicted_noise = unet(
            sample=latent_model_input,
            timestep=t,
            conditioning=conditioning,
        )

        latents = scheduler.step(predicted_noise, t, latents).prev_sample

    return latents

In [46]:
taef2_diffusers = (
    DiffusersTAEF2Wrapper(path="../../taef2.safetensors").eval().requires_grad_(False)
)
tiny_vae = taef2_diffusers.to(device, dtype)

In [47]:
with open("../../precomputes/flow_refractor/prompts.json") as f:
    prompts = json.load(f)


def build_random_prompt():
    if random.randint(0, 1) == 0:
        prompt = "Turn this image into style of "
        prompt += random.choice(prompts["general_styles"])
        prompt += "."
    else:
        prompt = "Turn this image into artwork in style of "
        prompt += random.choice(prompts["artist_names"])
        prompt += ", "
        num_modifiers = random.randint(1, 3)
        for i in range(num_modifiers):
            prompt += random.choice(prompts["artwork_styles"]) + ", "
        prompt = prompt[:-2] + "."

    return prompt

In [48]:
# prompt_idx = 0


# def build_random_prompt():
#     global prompt_idx
#     prompts = [
#         "Make cinematic light, professional high resolution golden hour photo, vibrant, UHD textures",
#         "Make it nighttime, beautiful neon streetlights of red and violet, professional, cyberpunk vibe",
#         "Make it grainy hsv camera photo, in style of old tv",
#         "Make these cubes black and white",
#         "Color all object to random different colors",
#         "Make it grayscale",
#         "Turn this into beautiful underwater photo, rays of light, HD, professional",
#         "Make it winter",
#         "Turn this into colorful art in style of Kandinsky, square and line textures on all surfaces",
#         "Turn this into retrowave art",
#     ]
#     prompt = prompts[prompt_idx]
#     prompt_idx += 1

#     return prompt


In [49]:
cap = cv2.VideoCapture("../../local_samples/videos/1.mp4")
# cap = cv2.VideoCapture(0)

skip = 2000

ret, frame = cap.read()

for i in range(skip):
    ret, frame = cap.read()


fourcc = cv2.VideoWriter_fourcc(*"mp4v")  # Codec for MP4
out = cv2.VideoWriter("output_video_2.mp4", fourcc, 30, (width, height))


while True:
    cond_latents, init_latents, final_latents = process_frame(
        frame, random.randint(0, 10000), build_random_prompt()
    )
    frame_A = cv2_to_np(resize_cv2(frame, height=height, width=width))

    interrupted = False
    for i in range(100):
        ret, frame = cap.read()
        frame_B = cv2_to_np(resize_cv2(frame, height=height, width=width))
        if not ret:
            break

        current_cond = cv2_to_np(resize_cv2(frame, height=height, width=width))
        current_cond = np_to_torch(current_cond)
        current_cond = norm_image(current_cond)
        current_cond, ids = prepare_image_latents(
            [current_cond], tiny_vae, batch_size=1, device=device, dtype=dtype
        )
        current_cond = unpack_latents_with_ids(current_cond, ids)
        current_cond = unpatchify_latents(current_cond)
        current_cond = current_cond.float()

        final_image_A = torch_to_cv2(
            denorm_image(decode_latents(final_latents, tiny_vae))
        )
        final_image_A = resize_cv2(final_image_A, height=128, width=240)

        final_B = refract(
            init=init_latents,
            cond_A=cond_latents,
            cond_B=current_cond,
            final_A=final_latents,
        )

        # recursive update
        cond_latents = current_cond
        final_latents = final_B
        ####

        final_image_B = torch_to_cv2(denorm_image(decode_latents(final_B, tiny_vae)))

        out_frame = final_image_B

        out_frame[:128, :240, :] = resize_cv2(frame, height=128, width=240)
        out_frame[:128, -240:, :] = final_image_A

        cv2.imshow("Video", out_frame)
        out.write(out_frame)
        if cv2.waitKey(1) & 0xFF == ord("q"):
            interrupted = True
            break

    if interrupted:
        break


cap.release()
out.release()
cv2.destroyAllWindows()